<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Annual_pr_2_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map.add("basemap_select")
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filterBounds(roi)
    .geometry().simplify(100)
)

vis_params = {'color': 'yellow'}
map.addLayer(border, vis_params, "border")

In [ ]:
gpm = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_RNL")
    .filterDate('2016-01-01','2025-12-31')
    .select('precipitation')
    .map(
        lambda img: img.clip(border).copyProperties(img, ['system:time_start'])
    )
)

In [ ]:
gpm.getInfo()

In [ ]:
import xarray as xr

In [ ]:
pr = xr.open_dataset(
    gpm,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.1,
    geometry = border
)
pr

In [ ]:
pr = pr.sortby("time") * 730.001

In [ ]:
pr

In [ ]:
import numpy as np

In [ ]:
pr_annual = pr.resample(time = 'YE').sum('time')

In [ ]:
pr_annual = xr.where(pr_annual == 0,  np.nan, pr_annual)

In [ ]:
import matplotlib.pyplot as plt

g = pr_annual.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    level = 20,
    cmap = "turbo_r"
)

for ax in g.axes.flat:
    ax.set_aspect('equal')

annual_change = pr_annual.diff('time')

In [ ]:
g = pr_annual.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    levels = 20,
    cmap = "turbo_r",
    cbar_kwargs = {'orientation': 'horizontal', 'label': 'Precipitation scale in mm/year', 'shrink': 0.4}
)

for ax in g.axes.flat:
    ax.set_aspect('equal')

plt.suptitle('Annual Precipitation for South Sudan: 2016-2025', y=1.02)

plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.show()


# Task
Calculate the mean annual precipitation over the `lon` and `lat` dimensions from the `pr_annual` xarray Dataset to obtain a single time series representing regional annual precipitation.

## Calculate Regional Annual Precipitation

### Subtask:
Compute the mean annual precipitation over the `lon` and `lat` dimensions from the `pr_annual` xarray Dataset to obtain a single time series representing regional annual precipitation.


**Reasoning**:
To compute the mean annual precipitation over the `lon` and `lat` dimensions, I will use the `.mean()` method on the `pr_annual` xarray Dataset, specifying the desired dimensions.



In [ ]:
regional_annual_precipitation = pr_annual.mean(dim = ['lon','lat'])
regional_annual_precipitation

## Prepare Data for Linear Regression

### Subtask:
Convert the regional annual precipitation time series into a pandas DataFrame, extracting the year as the independent variable and the mean precipitation as the dependent variable, suitable for training a linear regression model.


**Reasoning**:
First, I'll convert the `regional_annual_precipitation` xarray Dataset into a pandas DataFrame as instructed.



In [ ]:
df = regional_annual_precipitation.to_dataframe()
df

**Reasoning**:
Next, I will extract the year from the 'time' index, reset the DataFrame index, and select the 'year' and 'precipitation' columns to prepare the data for linear regression.



In [ ]:
df['year'] = df.index.year
df = df.reset_index()
df = df[['year', 'precipitation']]
df

## Train Linear Regression Model

### Subtask:
Train a linear regression model using the prepared historical annual precipitation data. This model will learn the trend in precipitation over the years.


**Reasoning**:
To begin training the linear regression model, I will import the necessary classes: `LinearRegression` for the model itself and `train_test_split` for potential data splitting (although not explicitly used in this step, it's a common import in model training scenarios).



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
print("Imported LinearRegression and train_test_split.")

**Reasoning**:
I will define the independent variable `X` as 'year' and the dependent variable `y` as 'precipitation' from the `df` DataFrame, reshape `X` for scikit-learn compatibility, then initialize and fit a `LinearRegression` model to the data.



In [ ]:
X = df['year']
y = df['precipitation']
X = X.values.reshape(-1, 1)

model = LinearRegression()
model.fit(X, y)

print("Linear regression model trained successfully.")

## Predict Precipitation for 2026

### Subtask:
Use the trained linear regression model to predict the regional annual precipitation for the year 2026.


#### Reasoning:
To predict the precipitation for 2026, I will create a NumPy array for the year 2026, reshape it for compatibility with the trained model, and then use the `model.predict()` method.

**Reasoning**:
I will create a NumPy array for the year 2026, reshape it to `(-1, 1)` for compatibility with the model, and then use the trained linear regression model to predict the precipitation for that year.



In [ ]:
year_2026 = np.array([2026]).reshape(-1, 1)
predicted_precipitation_2026 = model.predict(year_2026)
print(f"Predicted annual precipitation for 2026: {predicted_precipitation_2026[0]:.2f}")

## Visualize Prediction

### Subtask:
Generate a plot displaying the historical regional annual precipitation and the predicted precipitation value for 2026. Ensure the plot is clearly labeled with a legend.


**Reasoning**:
To create the visualization, I will import `matplotlib.pyplot` and `seaborn` to facilitate plotting the historical data, the linear regression line, and the predicted precipitation for 2026 with appropriate labels and a legend.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x=df['year'], y=df['precipitation'], label='Historical Data', color='blue', s=100)
plt.plot(df['year'], model.predict(X), color='red', label='Linear Regression Line')
plt.scatter(year_2026, predicted_precipitation_2026, color='green', marker='X', s=200, label='Predicted 2026 Precipitation')

plt.xlabel('Year')
plt.ylabel('Annual Precipitation (mm/year)')
plt.title('Regional Annual Precipitation: Historical and Predicted')
plt.legend()
plt.grid(True)
plt.show()
print("Plot generated showing historical data, regression line, and 2026 prediction.")

## Final Task

### Subtask:
Summarize the predicted annual precipitation for 2026 and present the visualization of the historical data and the forecast.


## Summary:

### Q&A
*   The predicted annual precipitation for 2026 is 680678.96 mm/year.

### Data Analysis Key Findings
*   Regional annual precipitation for the historical period (2016-2025) was calculated by averaging precipitation across longitude and latitude dimensions, resulting in a time series. For example, the first few values in this series were 7.007e+05 mm/year and 7.291e+05 mm/year.
*   A linear regression model was successfully trained using historical annual precipitation data (years 2016-2025) to model the trend over time.
*   The trained linear regression model predicted the annual precipitation for 2026 to be 680678.96 mm/year.
*   A visualization was generated, clearly showing the historical precipitation data points, the fitted linear regression line representing the trend, and the distinct predicted precipitation value for 2026.

### Insights or Next Steps
*   The linear regression model suggests a certain trend in annual precipitation; however, given that it's based on only 10 years of data (2016-2025), its predictive power might be limited.
*   Consider expanding the historical dataset or exploring more complex time series forecasting models (e.g., ARIMA, Prophet) for more robust and accurate long-term precipitation predictions, and evaluate model performance using metrics beyond visual inspection.
